In [2]:
from movie_recommender.models.popularity import PopularityRecommender
from movie_recommender.evaluation.metrics import hit_rate_k, ndcg_k
from movie_recommender.data.split import temporal_positive_split
import pandas as pd
from pathlib import Path

In [3]:
rating = pd.read_csv("../dataset/ml-latest-small/ratings.csv")

In [4]:
train, validation, test = temporal_positive_split(rating)

In [8]:
model = PopularityRecommender()
model.fit(train)

In [9]:
relevant_by_user = validation.groupby('userId')['movieId'].apply(set).to_dict()

In [10]:
print(relevant_by_user)

{1: {2012}, 2: {122882}, 3: {7899}, 4: {4896}, 5: {247}, 6: {1061}, 7: {48783}, 8: {11}, 9: {5965}, 10: {7154}, 11: {1584}, 12: {1721}, 13: {3827}, 14: {376}, 15: {84954}, 16: {91529}, 17: {111}, 18: {177593}, 19: {913}, 20: {1088}, 21: {47997}, 22: {3578}, 23: {6}, 24: {5791}, 25: {177593}, 26: {47}, 27: {2407}, 28: {316}, 29: {104841}, 30: {110}, 31: {1302}, 32: {1084}, 33: {919}, 34: {2443}, 35: {590}, 36: {3347}, 37: {1073}, 38: {246}, 39: {2391}, 40: {121}, 41: {6763}, 42: {2805}, 43: {1064}, 44: {889}, 45: {53121}, 46: {236}, 47: {1097}, 48: {5882}, 49: {79132}, 50: {1251}, 51: {60487}, 52: {50794}, 53: {1049}, 55: {4011}, 56: {69}, 57: {494}, 58: {708}, 59: {2726}, 60: {6016}, 61: {7361}, 62: {46976}, 63: {89904}, 64: {1345}, 65: {58559}, 66: {523}, 67: {99114}, 68: {117529}, 69: {4027}, 70: {1303}, 71: {1370}, 72: {648}, 73: {62434}, 74: {57147}, 75: {1240}, 76: {2318}, 77: {5378}, 78: {1270}, 79: {1127}, 80: {593}, 81: {589}, 82: {7325}, 83: {4975}, 84: {1380}, 85: {1237}, 86:

## since popularity model can recommend movies in train dataset, so exclude user with relevant movies not seen in train

In [11]:
train_catalog = set(train['movieId'])

In [12]:

evaluable_relevant_by_user = {user_id : relevant_movies 
                              for user_id, relevant_movies in relevant_by_user.items() 
                              if relevant_movies.issubset(train_catalog)}

In [13]:
excluded_users = (
    len(relevant_by_user)
    - len(evaluable_relevant_by_user)
)

In [15]:
print("Validation users:", len(relevant_by_user))
print("Evaluable users:", len(evaluable_relevant_by_user))
print("Excluded users:", excluded_users)

Validation users: 603
Evaluable users: 566
Excluded users: 37


# Evaluation of Popularity model on relevant movie from validation

In [17]:
evaluation_rows = []
for user_id, relevant_movies in evaluable_relevant_by_user.items():
    seen_movies = set(train.loc[train['userId']==user_id, 'movieId'])

    recommendations = model.recommend(seen_movies=seen_movies, k=10)

    hit_rate = hit_rate_k(recommendations, relevant_movies, k=10)
    ndcg = ndcg_k(recommendations, relevant_movies, k = 10)

    evaluation_rows.append({
        'userId': user_id,
        'relevant_movie': next(iter(relevant_movies)),
        'recommendations': recommendations,
        'hit_rate@10': hit_rate,
        'ndcg@10': ndcg
        })


In [20]:
per_user_results = pd.DataFrame(evaluation_rows)

In [21]:
per_user_results.head()

,userId,relevant_movie,recommendations,hit_rate@10,ndcg@10
0,1,2012,"[318, 296, 858, 589, 4993, 7153, 5952, 150, 27...",0.0,0.0
1,2,122882,"[318, 356, 296, 2571, 593, 260, 2959, 527, 119...",0.0,0.0
2,4,4896,"[318, 356, 296, 2571, 2959, 527, 110, 50, 1198...",0.0,0.0
3,5,247,"[318, 356, 2571, 593, 260, 2959, 1196, 1198, 8...",0.0,0.0
4,6,1061,"[296, 2571, 260, 2959, 527, 1196, 50, 1198, 85...",0.0,0.0


In [23]:
popularity_results = {
    "model": "popularity",
    "k": 10,
    "users_evaluated": len(per_user_results),
    "hit_rate": float(per_user_results["hit_rate@10"].mean()),
    "ndcg": float(per_user_results["ndcg@10"].mean()),
}

print(popularity_results)

{'model': 'popularity', 'k': 10, 'users_evaluated': 566, 'hit_rate': 0.045936395759717315, 'ndcg': 0.01998717919735628}


In [28]:
validation_results_path = Path('../reports/tables/validation_results.csv')
validation_results_path.parent.mkdir(parents=True, exist_ok=True)

validation_result = pd.DataFrame([popularity_results])
validation_result.to_csv(validation_results_path)

## Genre item features

In [20]:
from movie_recommender.features.item_features import build_genre_features, build_tag_features


ImportError: cannot import name 'build_tag_features' from 'movie_recommender.features.item_features' (C:\Users\bansh\Recommendation system\Movie_Recommendation_System\src\movie_recommender\features\item_features.py)

In [9]:
movies = pd.read_csv("../dataset/ml-latest-small/movies.csv")

In [12]:
tags = pd.read_csv("../dataset/ml-latest-small/tags.csv")

In [7]:
train_catalog = set(train['movieId'])

In [8]:
print(train_catalog)

{1, 2, 3, 32770, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 131098, 28, 29, 27, 31, 32, 30, 34, 131104, 36, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 50, 52, 53, 54, 55, 57, 58, 98361, 60, 61, 62, 65596, 131130, 63, 66, 65, 68, 69, 70, 72, 73, 74, 76, 77, 78, 79, 80, 81, 82, 85, 86, 163925, 88, 89, 92, 93, 94, 95, 65631, 97, 32862, 99, 100, 101, 102, 104, 105, 106, 107, 65642, 108, 110, 111, 112, 113, 116, 121, 122, 123, 32892, 125, 32898, 132, 135, 140, 141, 144, 145, 146, 65682, 147, 148, 150, 151, 32917, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 131237, 166, 168, 170, 171, 172, 173, 174, 175, 176, 177, 180, 181, 183, 184, 185, 186, 98491, 188, 187, 190, 191, 193, 194, 195, 196, 198, 199, 65738, 203, 204, 205, 65740, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 222, 223, 224, 225, 227, 229, 230, 231, 232, 233, 234, 235, 236, 237, 33004, 239, 238, 241, 242, 246, 247, 248, 249, 250, 252, 253, 255, 256, 257, 258

In [10]:
genre_features = build_genre_features(movies, train_catalog)

In [11]:

print("Feature matrix shape:", genre_features.shape)
genre_features.head()

Feature matrix shape: (6227, 19)


,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
movieId,,,,,,,,,,,,,,,,,,,
1,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [ ]:
tag_features = build_tag_features(
    tags=tags,
    catalog=train_catalog,
    min_document_frequency=2,
    max_features=1000,
)

print("Tag feature shape:", tag_features.shape)
print(
    "Movies with at least one tag feature:",
    int((tag_features.sum(axis=1) > 0).sum()),
)